# Querying PDF With Astra and LangChain

### A question-answering demo using Astra DB and LangChain, powered by Vector Search

#### Pre-requisites:

You need a **_Serverless Cassandra with Vector Search_** database on [Astra DB](https://astra.datastax.com) to run this demo. As outlined in more detail [here](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html#_prepare_for_using_your_vector_database), you should get a DB Token with role _Database Administrator_ and copy your Database ID: these connection parameters are needed momentarily.

You also need an [OpenAI API Key](https://cassio.org/start_here/#llm-access) for this demo to work.

#### What you will do:

- Setup: import dependencies, provide secrets, create the LangChain vector store;
- Run a Question-Answering loop retrieving the relevant headlines and having an LLM construct the answer.

Install the required dependencies:

In [20]:
!pip install -q cassio datasets langchain openai tiktoken

Import the packages you'll need:

In [ ]:
! pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.0 MB/s eta 0:00:00


In [21]:
# LangChain components to use
from langchain.vectorstores.cassandra import Cassandra
from langchain.indexes.vectorstore import VectorStoreIndexWrapper
from langchain.llms import OpenAI
from langchain.embeddings import OpenAIEmbeddings

# Support for dataset retrieval with Hugging Face
from datasets import load_dataset

# With CassIO, the engine powering the Astra DB integration in LangChain,
# you will also initialize the DB connection:
import cassio

In [22]:
!pip install PyPDF2

### Setup

In [35]:
import os

ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_ID = os.getenv("ASTRA_DB_ID")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


#### Provide your secrets:

Replace the following with your Astra DB connection details and your OpenAI API key:

In [27]:
from google.colab import drive
drive.mount('/content/drive')

from PyPDF2 import PdfReader

# provide the path of  pdf file/files.
pdfreader = PdfReader('/content/drive/MyDrive/Colab Notebooks/pdf_query/KEY-LLM-TERMS.pdf')


Mounted at /content/drive


In [28]:
from typing_extensions import Concatenate
# read text from pdf
raw_text = ''
for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [29]:
raw_text

'You may already know this, but just in case you\'re not familiar with the word "inference" that I use here:\nWhen working with Data Science models, you could be carrying out 2 very different \nactivities: training andinference .\n1. Training\nTraining is when you provide a model with data for it to adapt to get better at a task in the future. It does this by \nupdating its internal settings -the parameters or weights of the model. If you\'re Training a model that\'s already \nhad some training, the activity is called "fine -tuning".\n2. Inference\nInference is when you are working with a model that has already been trained . You are using that model to \nproduce new outputs on new inputs, taking advantage of everything it learned while it was being trained. \nInference is also sometimes referred to as "Execution" or "Running a model".\nAll of our use of APIs for GPT, Claude and Gemini in the last weeks are examples of inference . The "P" in GPT \nstands for "Pre -trained", meaning tha

Initialize the connection to your database:

_(do not worry if you see a few warnings, it's just that the drivers are chatty about negotiating protocol versions with the DB.)_

In [30]:
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)

Create the LangChain embedding and LLM objects for later usage:

In [37]:
!pip install -U langchain-openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.2 MB/s eta 0:00:00


In [38]:
from langchain_openai import OpenAI, OpenAIEmbeddings

# Then use them like normal
llm = OpenAI(openai_api_key=OPENAI_API_KEY)
embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

Create your LangChain vector store ... backed by Astra DB!

In [39]:
astra_vector_store = Cassandra(
    embedding=embedding,
    table_name="qa_mini_demo",
    session=None,
    keyspace=None,
)

In [40]:
from langchain.text_splitter import CharacterTextSplitter
# We need to split the text using Character Text Split such that it sshould not increse token size
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 800,
    chunk_overlap  = 200,
    length_function = len,
)
texts = text_splitter.split_text(raw_text)

In [41]:
texts[:50]

['You may already know this, but just in case you\'re not familiar with the word "inference" that I use here:\nWhen working with Data Science models, you could be carrying out 2 very different \nactivities: training andinference .\n1. Training\nTraining is when you provide a model with data for it to adapt to get better at a task in the future. It does this by \nupdating its internal settings -the parameters or weights of the model. If you\'re Training a model that\'s already \nhad some training, the activity is called "fine -tuning".\n2. Inference\nInference is when you are working with a model that has already been trained . You are using that model to \nproduce new outputs on new inputs, taking advantage of everything it learned while it was being trained.',
 'produce new outputs on new inputs, taking advantage of everything it learned while it was being trained. \nInference is also sometimes referred to as "Execution" or "Running a model".\nAll of our use of APIs for GPT, Claude an

### Load the dataset into the vector store



In [42]:

astra_vector_store.add_texts(texts[:50])

print("Inserted %i headlines." % len(texts[:50]))

astra_vector_index = VectorStoreIndexWrapper(vectorstore=astra_vector_store)

Inserted 50 headlines.


### Run the QA cycle

Simply run the cells and ask a question -- or `quit` to stop. (you can also stop execution with the "▪" button on the top toolbar)

Here are some suggested questions:
- _What is the difference between training and inference?_
- _What is the difference between model centric and business centric metrics_


In [43]:
first_question = True
while True:
    if first_question:
        query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\nWhat's your next question (or type 'quit' to exit): ").strip()

    if query_text.lower() == "quit":
        break

    if query_text == "":
        continue

    first_question = False

    print("\nQUESTION: \"%s\"" % query_text)
    answer = astra_vector_index.query(query_text, llm=llm).strip()
    print("ANSWER: \"%s\"\n" % answer)

    print("FIRST DOCUMENTS BY RELEVANCE:")
    for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print("    [%0.4f] \"%s ...\"" % (score, doc.page_content[:84]))


Enter your question (or type 'quit' to exit): What is the difference between training and inference? 

QUESTION: "What is the difference between training and inference?"


ANSWER: "Training is when a model is provided with data to adapt and improve its performance for future tasks. Inference is when a trained model is used to generate new outputs based on new inputs, utilizing what it has learned during training. Inference is also sometimes referred to as execution or running a model."

FIRST DOCUMENTS BY RELEVANCE:


    [0.9488] "You may already know this, but just in case you're not familiar with the word "infer ..."
    [0.9147] "produce new outputs on new inputs, taking advantage of everything it learned while i ..."
    [0.9072] "✅ Leaderboards rank LLMs based on benchmark results –Higher -ranked models generally ..."
    [0.8990] "Batch Size →Processing multiple requests at once can slow down response time. ○
Cont ..."

What's your next question (or type 'quit' to exit): What is the difference between business centric and model centric approaches in simple terms layman language

QUESTION: "What is the difference between business centric and model centric approaches in simple terms layman language"


ANSWER: "The difference between business centric and model centric approaches is in their focus and purpose. 

Business-centric metrics measure the real-world effectiveness of AI solutions, such as return on investment, cost savings, and user experience. They are used to determine if the AI solution is providing value to the business. 

On the other hand, model-centric metrics focus on the technical performance of the AI model itself, such as accuracy, loss, and perplexity. They are used by AI engineers to improve and optimize the model. 

In simpler terms, business-centric metrics look at how well the AI solution is working for the business, while model-centric metrics look at how well the AI model is performing."

FIRST DOCUMENTS BY RELEVANCE:


    [0.9190] "Business -Centric Metrics →Measure the real-world effectiveness of AI solutions ( RO ..."
    [0.9037] "Example in LLMs: If an AI chatbot answers user queries correctly 95% of the time , i ..."
    [0.8919] "For short queries (~1 -5 paragraphs), both models perform similarly , but for large  ..."
    [0.8895] "✅ Ideal for real -world AI applications where fast and scalable model serving is req ..."

What's your next question (or type 'quit' to exit): quit
